#Competencia Picas y Fijas

##Reglas


### Ambiente
* Al inicio del torneo el ambiente montará la unidad de Google Drive donde se encuentran los archivos .py de los agentes y ejecutará una competencia con una cantidad de rondas $N$ configurable mediante interfaz para definir al ganador.
  >🚨 El ambiente se encargará de verificar que los módulos e instancias de los agentes hereden correctamente de la interfaz requerida antes de iniciar la partida. Si hay errores de importación o falta de métodos, el agente será descalificado.

  >Esto se hará mediante la configuración inicial:
  ```
  Ambiente.setup
  ```
* En cada ronda el ambiente asumirá el rol de Juez Central, generando un único número secreto para ambos agentes, y llamará al método de inicialización de los agentes para que reseteen sus estados internos o bases de conocimiento.
  >🚨 El número generado por el ambiente siempre cumplirá estrictamente las reglas (4 dígitos enteros únicos entre 0 y 9). Si un agente falla al ejecutar su método de inicialización, esa ronda se dará por perdida para el agente infractor de forma inmediata.

  >Esto se hará al inicio de cada ronda mediante la función:
  ```
  Ambiente.initRound
  ```
  Cada turno el ambiente solicitará el intento de cada agente en paralelo mediante sus métodos de búsqueda, registrará ambos intentos y validará la estructura de la lista recibida.
  >🚨 Si la lista no tiene exactamente 4 enteros únicos entre 0 y 9, el intento se registrará de manera segura como "inválido" para ese turno, protegiendo la simulación de colapsos y penalizando al agente infractor en ese turno sin romper el juego.

  >Esto se hará cada turno mediante la función:
  ```
  Ambiente.getAttempts
  ```
 El ambiente evaluará internamente el intento del agente $A$ y el intento del agente $B$ comparándolos contra el número secreto central, calculando con precisión matemática las picas y fijas de cada uno.
  >🚨 El ambiente validará que la respuesta del método de evaluación sea una lista de 2 enteros $[P, F]$ donde $P + F \le 4$.
  
  >Esto se hará cada turno mediante la función:
  ```
  Ambiente.evaluateAttempts
  ```
 El ambiente entregará la respuesta procesada a cada agente respectivo para que actualice sus estados internos o bases de conocimiento con su propio resultado.
  >🚨  Si al finalizar el turno un agente obtiene la combinación $[0, 4]$, el ambiente declarará a ese agente como ganador de la ronda. En caso de que ambos obtengan $[0, 4]$ en el mismo turno, la ronda se declarará en empate.
  
  >Esto se hará cada turno mediante la función:
  ```
  Ambiente.dispatchFeedback
  ```
  Al finalizar las rondas configuradas, el ambiente contabilizará las victorias de cada agente para determinar el ganador definitivo del torneo y generará un reporte estadístico (porcentaje de victorias, turnos promedio y tiempo de ejecución).
  >🚨 Si al cabo de las $N$ rondas ambos agentes tienen la misma cantidad de victorias, el torneo finalizará en empate.


  > Esto se hará al concluir la simulación mediante la función:
  ```
  Ambiente.getWinner
  ```



###Juego
* Al inicio del jugo cada **agente** generara un numero aleatorio de cuatro cifras diferentes

  > 🚨 Los numeros 0000, 101 o 12345 no son numeros validos ya que, o repiten cifras, o no tienen el numero correcto de cifras.

  >Esto se hara mediante la funcion incial.

    ```
    TuAgente.start
    ```
* Cada agente intentará adivinar el número del Juez Central una vez por turno.

  >🚨Cada intento es una lista de enteros con 4 elementos (índices de 0 a 3) donde el elemento 0 es el primer dígito del número, así la lista [1,2,3,4] se lee 1234.

  >Esto se hará cada turno mediante la función:
    ```
    TuAgente.try
    ```
* Cada agente obtendrá retroalimentación de su intento cada turno por parte del ambiente, indicando qué tan acertado estuvo su intento. (La evaluación del contrincante ya no es responsabilidad del agente).

  >🚨 La retroalimentación vendrá dada por una lista de enteros con 2 elementos (índices de 0 a 1) donde picas y fijas son los elementos 0 y 1 respectivamente, así la lista [2,1] indica que el intento tuvo 2 picas y una fija.

  >Esto se hara cada **turno** mediante la funcion.

    ```
    TuAgente.feedBack(retroalimentacionLista)
    ```



###Interfaz para los modelos

In [ ]:
from abc import ABC, abstractmethod

class interfazAgente(ABC):

  @abstractmethod
  def start(self):
    """Función que inicializa o resetea el estado interno del agente al inicio de una nueva ronda"""
    pass

  @abstractmethod
  def try_attempt(self):
    """Función que, mediante los datos almacenados en su estado, intenta adivinar el número secreto del ambiente"""
    pass

  @abstractmethod
  def feedBack(self, retroalimentacionLista):
    """Función que recibe retroalimentacion (picas y fijas) del ambiente tras su último intento"""
    pass


##Ambiente

Conexion con la carpeta

In [2]:
import os
import sys

try:
    # --- Google Colab: montar Google Drive ---
    from google.colab import drive
    drive.mount("/content/drive")
    # IMPORTANTE: Reemplaza esta ruta por la carpeta exacta donde están tus archivos .py
    ruta_carpeta = "/content/drive/MyDrive/Competencia Picas y fijas"
except ImportError:
    # --- Ejecución local: no hay Drive que montar. La celda del torneo busca
    #     los agentes en la carpeta del notebook y en la que la contiene. ---
    ruta_carpeta = os.getcwd()
    print("ℹ️ Ejecución local (sin Google Colab).")

if ruta_carpeta not in sys.path:
    sys.path.append(ruta_carpeta)

print("📂 Carpeta de trabajo:", ruta_carpeta)


ℹ️ Ejecución local (sin Google Colab).
📂 Carpeta de trabajo: /home/sebastian/Documents/Systems_Engineering/SemesterIX/Sistemas-Inteligentes/Picas&Fijas/Ambientes


# Ambiente(código de ambiente juez)

In [ ]:
import time
import random
import os
import sys
import importlib
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# =========================================================================
# 1. INTERFAZ Y ADAPTADOR UNIVERSAL
# =========================================================================
class AdaptadorUniversal:
    def __init__(self, instancia_agente):
        self.agente = instancia_agente

    def start(self):
        if hasattr(self.agente, 'start'): self.agente.start()

    def try_attempt(self):
        if hasattr(self.agente, 'try_attempt'): return self.agente.try_attempt()
        if hasattr(self.agente, 'try_'): return getattr(self.agente, 'try_')()
        return [-1, -1, -1, -1]

    def receive_feedback(self, picas, fijas):
        if hasattr(self.agente, 'receive_feedback'):
            self.agente.receive_feedback(picas, fijas)
        elif hasattr(self.agente, 'feedBack'):
            self.agente.feedBack([picas, fijas])

# Agentes de Prueba Predeterminados
class AgenteAleatorio:
    def start(self): pass
    def try_attempt(self): return random.sample(range(10), 4)
    def receive_feedback(self, p, f): pass

class AgenteEstrategico:
    def __init__(self): self.opciones = []
    def start(self):
        import itertools
        self.opciones = [list(p) for p in itertools.permutations(range(10), 4)]
        self.ultimo_intento = None

    def try_attempt(self):
        self.ultimo_intento = self.opciones[0] if self.opciones else random.sample(range(10), 4)
        return self.ultimo_intento

    def receive_feedback(self, picas, fijas):
        if not self.ultimo_intento or not self.opciones: return
        calc_fb = lambda cand, int_: [sum(x in cand for x in int_) - sum(x == y for x, y in zip(cand, int_)), sum(x == y for x, y in zip(cand, int_))]
        self.opciones = [c for c in self.opciones if calc_fb(c, self.ultimo_intento) == [picas, fijas]]

# =========================================================================
# 2. SISTEMA DE TORNEO COMPLETO CON GUI
# =========================================================================
class TorneoPicasFijasUI:
    def __init__(self):
        self.agentes_disponibles = {
            "AgenteEstrategico": AgenteEstrategico,
            "AgenteAleatorio": AgenteAleatorio
        }
        self.agA, self.agB = None, None
        self.nombreA, self.nombreB = "", ""
        self.ronda_actual, self.max_rondas = 0, 3
        self.turno_actual = 0
        self.ronda_activa = False
        self.secreto_juez = []
        self.stats = {}
        self.tiempo_inicio_torneo = 0.0

        self.escanear_directorio()
        self.construir_interfaz()

    def escanear_directorio(self):
        sys.path.insert(0, '.')
        for arch in os.listdir('.'):
            if arch.endswith('.py') and not arch.startswith('.'):
                mod_name = arch[:-3]
                try:
                    if mod_name in sys.modules:
                        importlib.reload(sys.modules[mod_name])

                    mod = importlib.import_module(mod_name)
                    for k, v in mod.__dict__.items():
                        if isinstance(v, type) and k not in ["ABC", "InterfazAgente", "AdaptadorUniversal"]:
                            if hasattr(v, 'start') and (hasattr(v, 'try_attempt') or hasattr(v, 'try_')):
                                self.agentes_disponibles[f"{k} ({arch})"] = v
                except Exception as e:
                    pass

    def construir_interfaz(self):
        self.dd_agenteA = widgets.Dropdown(options=list(self.agentes_disponibles.keys()), description='Agente A:')
        self.dd_agenteB = widgets.Dropdown(options=list(self.agentes_disponibles.keys()), description='Agente B:')
        if len(self.agentes_disponibles) > 1: self.dd_agenteB.index = 1

        self.input_semilla = widgets.Text(value='', placeholder='Semilla opcional', description='🌱 Semilla:', layout=widgets.Layout(width='180px'))
        self.input_rondas = widgets.BoundedIntText(value=100, min=1, max=100000, description='🔢 Rondas:', layout=widgets.Layout(width='160px'))

        self.btn_refrescar = widgets.Button(description='🔄 Cargar .py', button_style='info')
        self.btn_preparar = widgets.Button(description='⚙️ Preparar Test', button_style='primary')
        self.btn_paso = widgets.Button(description='👣 Paso a Paso', button_style='warning', disabled=True)
        self.btn_auto = widgets.Button(description='🤖 Modo Auto', button_style='success', disabled=True)
        self.btn_masivo = widgets.Button(description='⚖️ Torneo Masivo', button_style='danger')

        self.chk_visual_masivo = widgets.Checkbox(
            value=False,
            description='🖨️ Modo Visual (imprime cada turno y el secreto de cada ronda)',
            indent=False,
            layout=widgets.Layout(width='420px')
        )

        self.btn_refrescar.on_click(self.on_refrescar)
        self.btn_preparar.on_click(self.on_preparar)
        self.btn_paso.on_click(self.on_paso)
        self.btn_auto.on_click(self.on_auto)
        self.btn_masivo.on_click(self.on_masivo)

        self.out_vis = widgets.Output()
        self.out_log = widgets.Output()

        display(widgets.VBox([
            widgets.HTML("<h2>⚔️ Torneo Competidor de Picas y Fijas</h2>"),
            widgets.HBox([self.dd_agenteA, self.dd_agenteB, self.input_semilla, self.btn_refrescar]),
            widgets.HBox([self.input_rondas, self.btn_preparar, self.btn_paso, self.btn_auto]),
            widgets.HBox([self.btn_masivo, self.chk_visual_masivo]),
            self.out_vis, self.out_log
        ]))

    def on_refrescar(self, _):
        self.escanear_directorio()
        opts = list(self.agentes_disponibles.keys())
        self.dd_agenteA.options, self.dd_agenteB.options = opts, opts
        with self.out_log: print("🔄 Agentes escaneados y actualizados.")

    def reset_stats(self):
        self.stats = {
            "vA": 0, "vB": 0, "empates": 0,
            "turnosA": [], "turnosB": [],
            "tiempoA": 0.0, "tiempoB": 0.0, "movsA": 0, "movsB": 0,
            "erroresA": 0, "erroresB": 0
        }
        self.historial_secretos = []

    def aplicar_semilla(self):
        semilla_val = self.input_semilla.value.strip()
        if semilla_val:
            random.seed(semilla_val)
            return f"🌱 Semilla activa: '{semilla_val}' (Modo Determinista)"
        else:
            random.seed()
            return "🎲 Semilla: Aleatoria"

    def on_preparar(self, _):
        self.nombreA = self.dd_agenteA.value.split(' ')[0]
        self.nombreB = self.dd_agenteB.value.split(' ')[0]
        self.claseA = self.agentes_disponibles[self.dd_agenteA.value]
        self.claseB = self.agentes_disponibles[self.dd_agenteB.value]

        self.max_rondas = self.input_rondas.value
        self.ronda_actual = 0
        self.reset_stats()

        msg_semilla = self.aplicar_semilla()
        self.btn_paso.disabled, self.btn_auto.disabled = False, False
        self.tiempo_inicio_torneo = time.time()

        with self.out_log:
            clear_output()
            print(f"✅ Enfrentamiento Configurado: {self.nombreA} vs {self.nombreB}")
            print(f"   {msg_semilla}")

        self.iniciar_nueva_ronda()

    def iniciar_nueva_ronda(self):
        self.ronda_actual += 1
        self.turno_actual = 0
        self.ronda_activa = True

        self.secreto_juez = random.sample(range(10), 4)
        self.historial_secretos.append(self.secreto_juez)

        self.agA = AdaptadorUniversal(self.claseA())
        self.agB = AdaptadorUniversal(self.claseB())
        self.agA.start()
        self.agB.start()

        self.actualizar_vis()
        with self.out_log:
            print(f"\n==================================================")
            print(f"--- INICIANDO RONDA {self.ronda_actual} DE {self.max_rondas} ---")
            print(f"🎯 Número Secreto de la Ronda (elegido por el Juez): {self.secreto_juez}")
            print(f"==================================================")

    def actualizar_vis(self):
        with self.out_vis:
            clear_output(wait=True)
            str_historial = " | ".join([f"R{i+1}: {sec}" for i, sec in enumerate(self.historial_secretos)])
            tiempo_transcurrido = time.time() - self.tiempo_inicio_torneo

            html = f"""
            <div style="background-color: #f8f9fa; padding: 15px; border-radius: 8px; border: 2px solid #6c757d; margin: 10px 0;">
                <h3 style="margin-top: 0; color: #343a40;">👁️ PANEL ESPECTADOR</h3>
                <p style="font-size: 16px;"><b>🎯 Número Secreto de la Ronda Actual ({self.ronda_actual}):</b>
                   <span style="color: #d9534f; font-family: monospace; font-size: 20px; font-weight: bold; letter-spacing: 3px;">{self.secreto_juez}</span>
                </p>
                <p style="font-size: 14px; color: #555;"><b>📜 Historial de Secretos Guardados:</b> {str_historial}</p>
                <hr style="border-top: 1px solid #ccc;">
                <p><b>⏱️ Cronómetro Global:</b> {tiempo_transcurrido:.2f} s | <b>Ronda:</b> {self.ronda_actual} / {self.max_rondas} | <b>Turno:</b> {self.turno_actual}</p>
                <p><b>🏆 Marcador:</b> {self.nombreA} ({self.stats['vA']}) vs {self.nombreB} ({self.stats['vB']}) | Empates: {self.stats['empates']}</p>
            </div>
            """
            display(widgets.HTML(html))

    def evaluar_intento(self, intento):
        try:
            if len(intento) != 4 or len(set(intento)) != 4 or not all(isinstance(x, int) and 0 <= x <= 9 for x in intento):
                return False, [0, 0]
        except: return False, [0, 0]

        p = sum(x in self.secreto_juez for x in intento)
        f = sum(x == y for x, y in zip(self.secreto_juez, intento))
        return True, [p - f, f]

    def ejecutar_un_turno(self, silencioso=False):
        if not self.ronda_activa:
            if self.ronda_actual < self.max_rondas:
                self.iniciar_nueva_ronda()
                return True
            else:
                return False

        self.turno_actual += 1

        t0 = time.perf_counter()
        intA = self.agA.try_attempt()
        self.stats["tiempoA"] += (time.perf_counter() - t0)
        self.stats["movsA"] += 1

        t0 = time.perf_counter()
        intB = self.agB.try_attempt()
        self.stats["tiempoB"] += (time.perf_counter() - t0)
        self.stats["movsB"] += 1

        validoA, fbA = self.evaluar_intento(intA)
        validoB, fbB = self.evaluar_intento(intB)

        if not validoA: self.stats["erroresA"] += 1
        if not validoB: self.stats["erroresB"] += 1

        self.agA.receive_feedback(*fbA)
        self.agB.receive_feedback(*fbB)

        winA, winB = (fbA == [0, 4]), (fbB == [0, 4])

        if winA and winB:
            self.stats["empates"] += 1
            self.stats["turnosA"].append(self.turno_actual)
            self.stats["turnosB"].append(self.turno_actual)
            self.ronda_activa = False
        elif winA:
            self.stats["vA"] += 1
            self.stats["turnosA"].append(self.turno_actual)
            self.ronda_activa = False
        elif winB:
            self.stats["vB"] += 1
            self.stats["turnosB"].append(self.turno_actual)
            self.ronda_activa = False

        if not silencioso:
            self.actualizar_vis()
            with self.out_log:
                print(f"\n--- Turno {self.turno_actual} ---")
                print(f'"{self.nombreA}" intenta: {intA} -> (Picas: {fbA[0]}, Fijas: {fbA[1]})')
                print(f'"{self.nombreB}" intenta: {intB} -> (Picas: {fbB[0]}, Fijas: {fbB[1]})')

                if not self.ronda_activa:
                    if winA and winB: print(f"\n🤝 ¡EMPATARON LA RONDA {self.ronda_actual}!")
                    elif winA: print(f"\n🏆 ¡{self.nombreA} GANÓ LA RONDA {self.ronda_actual}!")
                    elif winB: print(f"\n🏆 ¡{self.nombreB} GANÓ LA RONDA {self.ronda_actual}!")

        if not self.ronda_activa and self.ronda_actual == self.max_rondas:
            return False
        return True

    def on_paso(self, _):
        if not self.ejecutar_un_turno(silencioso=False): self.finalizar_torneo()

    def on_auto(self, _):
        self.btn_paso.disabled, self.btn_auto.disabled = True, True
        while self.ejecutar_un_turno(silencioso=False):
            time.sleep(0.1)
        self.finalizar_torneo()

    def on_masivo(self, _):
        self.nombreA = self.dd_agenteA.value.split(' ')[0]
        self.nombreB = self.dd_agenteB.value.split(' ')[0]
        self.claseA = self.agentes_disponibles[self.dd_agenteA.value]
        self.claseB = self.agentes_disponibles[self.dd_agenteB.value]

        self.max_rondas = self.input_rondas.value
        self.reset_stats()
        msg_semilla = self.aplicar_semilla()
        self.btn_paso.disabled, self.btn_auto.disabled = True, True
        self.tiempo_inicio_torneo = time.time()
        modo_visual = self.chk_visual_masivo.value

        with self.out_log:
            clear_output()
            print(f"🚀 Ejecutando Torneo Masivo ({self.max_rondas} rondas)...")
            print(f"   {msg_semilla}")

        # PREPARAR BARRA DE PROGRESO
        with self.out_vis:
            clear_output(wait=True)
            if not modo_visual:
                self.barra_progreso = widgets.IntProgress(value=0, min=0, max=self.max_rondas, bar_style='info', layout=widgets.Layout(width='80%'))
                self.label_progreso = widgets.Label(value=f"0 / {self.max_rondas} Rondas (0%)")
                display(widgets.VBox([
                    widgets.HTML(f"<h3>⏳ Simulando {self.max_rondas} partidas en segundo plano...</h3>"),
                    widgets.HBox([self.barra_progreso, self.label_progreso])
                ]))

        # Calcular cada cuánto actualizar la barra (máximo 100 actualizaciones para no ralentizar el navegador)
        paso_actualizacion = max(1, self.max_rondas // 100)

        for r in range(1, self.max_rondas + 1):
            self.ronda_actual = r
            self.turno_actual = 0
            self.ronda_activa = True

            self.secreto_juez = random.sample(range(10), 4)
            self.historial_secretos.append(self.secreto_juez)
            self.agA = AdaptadorUniversal(self.claseA())
            self.agB = AdaptadorUniversal(self.claseB())
            self.agA.start()
            self.agB.start()

            if modo_visual:
                self.actualizar_vis()
                with self.out_log:
                    print(f"\n==================================================")
                    print(f"--- INICIANDO RONDA {self.ronda_actual} DE {self.max_rondas} ---")
                    print(f"🎯 Número Secreto: {self.secreto_juez}")
                    print(f"==================================================")

            while self.ronda_activa:
                self.ejecutar_un_turno(silencioso=not modo_visual)

            # ACTUALIZAR BARRA DE PROGRESO (Sin recargar toda la interfaz)
            if not modo_visual and (r % paso_actualizacion == 0 or r == self.max_rondas):
                self.barra_progreso.value = r
                pct = int((r / self.max_rondas) * 100)
                self.label_progreso.value = f"{r} / {self.max_rondas} Rondas ({pct}%)"

        self.finalizar_torneo()

    def finalizar_torneo(self):
        self.btn_paso.disabled, self.btn_auto.disabled = True, True
        tiempo_total = time.time() - self.tiempo_inicio_torneo

        prom_A = sum(self.stats['turnosA'])/len(self.stats['turnosA']) if self.stats['turnosA'] else 0
        prom_B = sum(self.stats['turnosB'])/len(self.stats['turnosB']) if self.stats['turnosB'] else 0
        pct_A = (self.stats['vA'] / self.max_rondas) * 100
        pct_B = (self.stats['vB'] / self.max_rondas) * 100
        ms_A = (self.stats['tiempoA'] / self.stats['movsA']) * 1000 if self.stats['movsA'] else 0
        ms_B = (self.stats['tiempoB'] / self.stats['movsB']) * 1000 if self.stats['movsB'] else 0

        with self.out_log:
            print("\n" + "="*60)
            print("🏁 REPORTE FINAL DEL TORNEO")
            print("="*60)
            print(f"⏱️ Tiempo Total de Ejecución: {tiempo_total:.2f} segundos")
            print(f"\n📊 VICTORIAS:")
            print(f"  • {self.nombreA}: {self.stats['vA']} ({pct_A:.1f}%)")
            print(f"  • {self.nombreB}: {self.stats['vB']} ({pct_B:.1f}%)")
            print(f"  • Empates: {self.stats['empates']}")
            print(f"\n🎯 TURNOS PARA GANAR (PROMEDIO):")
            print(f"  • {self.nombreA}: {prom_A:.2f} turnos")
            print(f"  • {self.nombreB}: {prom_B:.2f} turnos")
            print(f"\n⚡ TIEMPO DE CÓMPUTO:")
            print(f"  • {self.nombreA}: {ms_A:.3f} ms / turno")
            print(f"  • {self.nombreB}: {ms_B:.3f} ms / turno")

            print("\n" + "🌟"*25)
            if self.stats['vA'] > self.stats['vB']: print(f"🏅 GANADOR: ¡{self.nombreA.upper()}!")
            elif self.stats['vB'] > self.stats['vA']: print(f"🏅 GANADOR: ¡{self.nombreB.upper()}!")
            else: print(f"⚖️ ¡EMPATE!")
            print("🌟"*25 + "\n")

        with self.out_vis:
            clear_output(wait=True)
            fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))
            ax1.bar([self.nombreA, self.nombreB, "Empates"], [self.stats['vA'], self.stats['vB'], self.stats['empates']], color=['#2196F3', '#FF9800', '#9E9E9E'])
            ax1.set_title("Rondas Ganadas")
            ax2.bar([self.nombreA, self.nombreB], [prom_A, prom_B], color=['#4CAF50', '#8BC34A'])
            ax2.set_title("Turnos promedio")
            ax3.bar([self.nombreA, self.nombreB], [ms_A, ms_B], color=['#9C27B0', '#E91E63'])
            ax3.set_title("Tiempo (ms/turno)")
            plt.tight_layout()
            plt.show()

# Instanciar el GUI
UI = TorneoPicasFijasUI()

In [1]:
import time
import random
import os
import sys
import importlib
import itertools
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# =========================================================================
# 0. DE DÓNDE SE CARGAN LOS AGENTES
# =========================================================================
# Local: la carpeta desde la que corre el notebook (Ambientes/) y la que la
#        contiene, que es donde están los .py de los agentes del repo.
# Colab: la carpeta de Google Drive de la competencia, si está montada.
CARPETA_AGENTES_DRIVE = "/content/drive/MyDrive/Competencia Picas y fijas/"
CARPETAS_AGENTES_EXTRA = []   # rutas extra, si guardas los agentes en otro sitio
ARCHIVO_MARCA = "Picas_Y_Fijas_Agent_Compute.py"   # identifica la carpeta local


def carpetas_de_agentes():
    """Carpetas existentes (sin repetir) donde buscar los .py de los agentes:
    la carpeta actual, la que la contiene si ahí está ARCHIVO_MARCA (el
    notebook vive en Ambientes/) y la de Drive si Colab la montó."""
    aqui = os.path.abspath(os.getcwd())
    padre = os.path.dirname(aqui)
    rutas = list(CARPETAS_AGENTES_EXTRA) + [aqui]
    if os.path.exists(os.path.join(padre, ARCHIVO_MARCA)):
        rutas.append(padre)
    rutas.append(CARPETA_AGENTES_DRIVE)

    vistas, salida = set(), []
    for ruta in rutas:
        ruta = os.path.abspath(ruta)
        if ruta not in vistas and os.path.isdir(ruta):
            vistas.add(ruta)
            salida.append(ruta)
    return salida


# =========================================================================
# 1. INTERFAZ Y ADAPTADOR UNIVERSAL
# =========================================================================
class AdaptadorUniversal:
    def __init__(self, instancia_agente):
        self.agente = instancia_agente

    def compute(self, feedback):
        if hasattr(self.agente, 'compute'):
            return self.agente.compute(feedback)
        return [-1, -1, -1, -1]

# Agentes de Prueba Predeterminados
class AgenteAleatorio:
    def compute(self, feedback):
        # Recibe [picas, fijas]. En el primer turno recibe [-1, -1]
        return random.sample(range(10), 4)

class AgenteEstrategico:
    def __init__(self):
        self.opciones = [list(p) for p in itertools.permutations(range(10), 4)]
        self.ultimo_intento = None

    def compute(self, feedback):
        picas, fijas = feedback
        # Si existe retroalimentación válida de un turno previo, filtra el espacio de búsqueda
        if picas != -1 and fijas != -1 and self.ultimo_intento and self.opciones:
            calc_fb = lambda cand, int_: [
                sum(x in cand for x in int_) - sum(x == y for x, y in zip(cand, int_)),
                sum(x == y for x, y in zip(cand, int_))
            ]
            self.opciones = [c for c in self.opciones if calc_fb(c, self.ultimo_intento) == [picas, fijas]]

        self.ultimo_intento = self.opciones[0] if self.opciones else random.sample(range(10), 4)
        return self.ultimo_intento

# =========================================================================
# 2. SISTEMA DE TORNEO COMPLETO CON GUI
# =========================================================================
class TorneoPicasFijasUI:
    def __init__(self):
        self.agentes_disponibles = {
            "AgenteEstrategico": AgenteEstrategico,
            "AgenteAleatorio": AgenteAleatorio
        }
        self.agA, self.agB = None, None
        self.fbA, self.fbB = [-1, -1], [-1, -1]
        self.nombreA, self.nombreB = "", ""
        self.ronda_actual, self.max_rondas = 0, 3
        self.turno_actual = 0
        self.ronda_activa = False
        self.secreto_juez = []
        self.stats = {}
        self.tiempo_inicio_torneo = 0.0

        self.escanear_directorio()
        self.construir_interfaz()

    def escanear_directorio(self):
        carpetas = carpetas_de_agentes()

        if not carpetas:
            print("⚠️ No se encontró ninguna carpeta de agentes. Revisa CARPETA_AGENTES_DRIVE / CARPETAS_AGENTES_EXTRA.")
            return

        archivos_encontrados = 0

        for ruta_carpeta in carpetas:
            print(f"📂 Buscando agentes en '{ruta_carpeta}'")

            if ruta_carpeta not in sys.path:
                sys.path.insert(0, ruta_carpeta)

            for arch in sorted(os.listdir(ruta_carpeta)):
                if arch.endswith('.py') and not arch.startswith('.'):
                    archivos_encontrados += 1
                    mod_name = arch[:-3]
                    try:
                        if mod_name in sys.modules:
                            importlib.reload(sys.modules[mod_name])

                        mod = importlib.import_module(mod_name)

                        clases_encontradas = 0
                        for k, v in mod.__dict__.items():
                            if isinstance(v, type) and k not in ["ABC", "InterfazAgente", "AdaptadorUniversal"]:
                                # Se verifica si la clase proviene del mismo archivo (evita clases importadas)
                                if getattr(v, '__module__', None) == mod_name:
                                    if hasattr(v, 'compute'):
                                        self.agentes_disponibles[f"{k} ({arch})"] = v
                                        clases_encontradas += 1
                                        print(f"✅ Agente cargado con éxito: {k} desde {arch}")
                                    else:
                                        print(f"⚠️ La clase '{k}' en '{arch}' fue ignorada porque NO tiene el método 'compute'.")

                        if clases_encontradas == 0:
                            print(f"ℹ️ No se encontraron clases válidas con método 'compute' en '{arch}'.")

                    except Exception as e:
                        print(f"❌ Error al importar '{arch}': {type(e).__name__} - {e}")

        if archivos_encontrados == 0:
            print(f"⚠️ No se encontraron archivos .py en: {carpetas}")

    def construir_interfaz(self):
        self.dd_agenteA = widgets.Dropdown(options=list(self.agentes_disponibles.keys()), description='Agente A:')
        self.dd_agenteB = widgets.Dropdown(options=list(self.agentes_disponibles.keys()), description='Agente B:')
        if len(self.agentes_disponibles) > 1: self.dd_agenteB.index = 1

        self.input_semilla = widgets.Text(value='', placeholder='Semilla opcional', description='🌱 Semilla:', layout=widgets.Layout(width='180px'))
        self.input_rondas = widgets.BoundedIntText(value=100, min=1, max=100000, description='🔢 Rondas:', layout=widgets.Layout(width='160px'))

        self.btn_refrescar = widgets.Button(description='🔄 Cargar .py', button_style='info')
        self.btn_preparar = widgets.Button(description='⚙️ Preparar Test', button_style='primary')
        self.btn_paso = widgets.Button(description='👣 Paso a Paso', button_style='warning', disabled=True)
        self.btn_auto = widgets.Button(description='🤖 Modo Auto', button_style='success', disabled=True)
        self.btn_masivo = widgets.Button(description='⚖️ Torneo Masivo', button_style='danger')

        self.chk_visual_masivo = widgets.Checkbox(
            value=False,
            description='🖨️ Modo Visual (imprime cada turno y el secreto de cada ronda)',
            indent=False,
            layout=widgets.Layout(width='420px')
        )

        self.btn_refrescar.on_click(self.on_refrescar)
        self.btn_preparar.on_click(self.on_preparar)
        self.btn_paso.on_click(self.on_paso)
        self.btn_auto.on_click(self.on_auto)
        self.btn_masivo.on_click(self.on_masivo)

        self.out_vis = widgets.Output()
        self.out_log = widgets.Output()

        display(widgets.VBox([
            widgets.HTML("<h2>⚔️ Torneo Competidor de Picas y Fijas</h2>"),
            widgets.HBox([self.dd_agenteA, self.dd_agenteB, self.input_semilla, self.btn_refrescar]),
            widgets.HBox([self.input_rondas, self.btn_preparar, self.btn_paso, self.btn_auto]),
            widgets.HBox([self.btn_masivo, self.chk_visual_masivo]),
            self.out_vis, self.out_log
        ]))

    def on_refrescar(self, _):
        self.escanear_directorio()
        opts = list(self.agentes_disponibles.keys())
        self.dd_agenteA.options, self.dd_agenteB.options = opts, opts
        with self.out_log: print("🔄 Agentes escaneados y actualizados.")

    def reset_stats(self):
        self.stats = {
            "vA": 0, "vB": 0, "empates": 0,
            "turnosA": [], "turnosB": [],
            "tiempoA": 0.0, "tiempoB": 0.0, "movsA": 0, "movsB": 0,
            "erroresA": 0, "erroresB": 0
        }
        self.historial_secretos = []

    def aplicar_semilla(self):
        semilla_val = self.input_semilla.value.strip()
        if semilla_val:
            random.seed(semilla_val)
            return f"🌱 Semilla activa: '{semilla_val}' (Modo Determinista)"
        else:
            random.seed()
            return "🎲 Semilla: Aleatoria"

    def on_preparar(self, _):
        self.nombreA = self.dd_agenteA.value.split(' ')[0]
        self.nombreB = self.dd_agenteB.value.split(' ')[0]
        self.claseA = self.agentes_disponibles[self.dd_agenteA.value]
        self.claseB = self.agentes_disponibles[self.dd_agenteB.value]

        self.max_rondas = self.input_rondas.value
        self.ronda_actual = 0
        self.reset_stats()

        msg_semilla = self.aplicar_semilla()
        self.btn_paso.disabled, self.btn_auto.disabled = False, False
        self.tiempo_inicio_torneo = time.time()

        with self.out_log:
            clear_output()
            print(f"✅ Enfrentamiento Configurado: {self.nombreA} vs {self.nombreB}")
            print(f"   {msg_semilla}")

        self.iniciar_nueva_ronda()

    def iniciar_nueva_ronda(self):
        self.ronda_actual += 1
        self.turno_actual = 0
        self.ronda_activa = True

        # El ambiente genera de forma independiente el número secreto
        self.secreto_juez = random.sample(range(10), 4)
        self.historial_secretos.append(self.secreto_juez)

        self.agA = AdaptadorUniversal(self.claseA())
        self.agB = AdaptadorUniversal(self.claseB())

        # Inicialización del feedback en [-1, -1] para indicar que es el primer turno
        self.fbA = [-1, -1]
        self.fbB = [-1, -1]

        self.actualizar_vis()
        with self.out_log:
            print(f"\n==================================================")
            print(f"--- INICIANDO RONDA {self.ronda_actual} DE {self.max_rondas} ---")
            print(f"🎯 Número Secreto de la Ronda (Generado por el Ambiente): {self.secreto_juez}")
            print(f"==================================================")

    def actualizar_vis(self):
        with self.out_vis:
            clear_output(wait=True)
            str_historial = " | ".join([f"R{i+1}: {sec}" for i, sec in enumerate(self.historial_secretos)])
            tiempo_transcurrido = time.time() - self.tiempo_inicio_torneo

            html = f"""
            <div style="background-color: #f8f9fa; padding: 15px; border-radius: 8px; border: 2px solid #6c757d; margin: 10px 0;">
                <h3 style="margin-top: 0; color: #343a40;">👁️ PANEL ESPECTADOR</h3>
                <p style="font-size: 16px;"><b>🎯 Número Secreto de la Ronda Actual ({self.ronda_actual}):</b>
                   <span style="color: #d9534f; font-family: monospace; font-size: 20px; font-weight: bold; letter-spacing: 3px;">{self.secreto_juez}</span>
                </p>
                <p style="font-size: 14px; color: #555;"><b>📜 Historial de Secretos Guardados:</b> {str_historial}</p>
                <hr style="border-top: 1px solid #ccc;">
                <p><b>⏱️ Cronómetro Global:</b> {tiempo_transcurrido:.2f} s | <b>Ronda:</b> {self.ronda_actual} / {self.max_rondas} | <b>Turno:</b> {self.turno_actual}</p>
                <p><b>🏆 Marcador:</b> {self.nombreA} ({self.stats['vA']}) vs {self.nombreB} ({self.stats['vB']}) | Empates: {self.stats['empates']}</p>
            </div>
            """
            display(widgets.HTML(html))

    def evaluar_intento(self, intento):
        try:
            if len(intento) != 4 or len(set(intento)) != 4 or not all(isinstance(x, int) and 0 <= x <= 9 for x in intento):
                return False, [0, 0]
        except Exception:
            return False, [0, 0]

        p = sum(x in self.secreto_juez for x in intento)
        f = sum(x == y for x, y in zip(self.secreto_juez, intento))
        return True, [p - f, f]

    def ejecutar_un_turno(self, silencioso=False):
        if not self.ronda_activa:
            if self.ronda_actual < self.max_rondas:
                self.iniciar_nueva_ronda()
                return True
            else:
                return False

        self.turno_actual += 1

        # Comunicación estricta mediante compute([picas, fijas])
        t0 = time.perf_counter()
        intA = self.agA.compute(self.fbA)
        self.stats["tiempoA"] += (time.perf_counter() - t0)
        self.stats["movsA"] += 1

        t0 = time.perf_counter()
        intB = self.agB.compute(self.fbB)
        self.stats["tiempoB"] += (time.perf_counter() - t0)
        self.stats["movsB"] += 1

        validoA, fbA_nuevo = self.evaluar_intento(intA)
        validoB, fbB_nuevo = self.evaluar_intento(intB)

        if not validoA: self.stats["erroresA"] += 1
        if not validoB: self.stats["erroresB"] += 1

        # Actualizar la retroalimentación que recibirán los agentes en la siguiente iteración
        self.fbA = fbA_nuevo
        self.fbB = fbB_nuevo

        winA, winB = (fbA_nuevo == [0, 4]), (fbB_nuevo == [0, 4])

        if winA and winB:
            self.stats["empates"] += 1
            self.stats["turnosA"].append(self.turno_actual)
            self.stats["turnosB"].append(self.turno_actual)
            self.ronda_activa = False
        elif winA:
            self.stats["vA"] += 1
            self.stats["turnosA"].append(self.turno_actual)
            self.ronda_activa = False
        elif winB:
            self.stats["vB"] += 1
            self.stats["turnosB"].append(self.turno_actual)
            self.ronda_activa = False

        if not silencioso:
            self.actualizar_vis()
            with self.out_log:
                print(f"\n--- Turno {self.turno_actual} ---")
                print(f'"{self.nombreA}" intenta: {intA} -> (Picas: {fbA_nuevo[0]}, Fijas: {fbA_nuevo[1]})')
                print(f'"{self.nombreB}" intenta: {intB} -> (Picas: {fbB_nuevo[0]}, Fijas: {fbB_nuevo[1]})')

                if not self.ronda_activa:
                    if winA and winB: print(f"\n🤝 ¡EMPATARON LA RONDA {self.ronda_actual}!")
                    elif winA: print(f"\n🏆 ¡{self.nombreA} GANÓ LA RONDA {self.ronda_actual}!")
                    elif winB: print(f"\n🏆 ¡{self.nombreB} GANÓ LA RONDA {self.ronda_actual}!")

        if not self.ronda_activa and self.ronda_actual == self.max_rondas:
            return False
        return True

    def on_paso(self, _):
        if not self.ejecutar_un_turno(silencioso=False): self.finalizar_torneo()

    def on_auto(self, _):
        self.btn_paso.disabled, self.btn_auto.disabled = True, True
        while self.ejecutar_un_turno(silencioso=False):
            time.sleep(0.1)
        self.finalizar_torneo()

    def on_masivo(self, _):
        self.nombreA = self.dd_agenteA.value.split(' ')[0]
        self.nombreB = self.dd_agenteB.value.split(' ')[0]
        self.claseA = self.agentes_disponibles[self.dd_agenteA.value]
        self.claseB = self.agentes_disponibles[self.dd_agenteB.value]

        self.max_rondas = self.input_rondas.value
        self.reset_stats()
        msg_semilla = self.aplicar_semilla()
        self.btn_paso.disabled, self.btn_auto.disabled = True, True
        self.tiempo_inicio_torneo = time.time()
        modo_visual = self.chk_visual_masivo.value

        with self.out_log:
            clear_output()
            print(f"🚀 Ejecutando Torneo Masivo ({self.max_rondas} rondas)...")
            print(f"   {msg_semilla}")

        with self.out_vis:
            clear_output(wait=True)
            if not modo_visual:
                self.barra_progreso = widgets.IntProgress(value=0, min=0, max=self.max_rondas, bar_style='info', layout=widgets.Layout(width='80%'))
                self.label_progreso = widgets.Label(value=f"0 / {self.max_rondas} Rondas (0%)")
                display(widgets.VBox([
                    widgets.HTML(f"<h3>⏳ Simulando {self.max_rondas} partidas en segundo plano...</h3>"),
                    widgets.HBox([self.barra_progreso, self.label_progreso])
                ]))

        paso_actualizacion = max(1, self.max_rondas // 100)

        for r in range(1, self.max_rondas + 1):
            self.ronda_actual = r
            self.turno_actual = 0
            self.ronda_activa = True

            self.secreto_juez = random.sample(range(10), 4)
            self.historial_secretos.append(self.secreto_juez)
            self.agA = AdaptadorUniversal(self.claseA())
            self.agB = AdaptadorUniversal(self.claseB())
            self.fbA = [-1, -1]
            self.fbB = [-1, -1]

            if modo_visual:
                self.actualizar_vis()
                with self.out_log:
                    print(f"\n==================================================")
                    print(f"--- INICIANDO RONDA {self.ronda_actual} DE {self.max_rondas} ---")
                    print(f"🎯 Número Secreto: {self.secreto_juez}")
                    print(f"==================================================")

            while self.ronda_activa:
                self.ejecutar_un_turno(silencioso=not modo_visual)

            if not modo_visual and (r % paso_actualizacion == 0 or r == self.max_rondas):
                self.barra_progreso.value = r
                pct = int((r / self.max_rondas) * 100)
                self.label_progreso.value = f"{r} / {self.max_rondas} Rondas ({pct}%)"

        self.finalizar_torneo()

    def finalizar_torneo(self):
        self.btn_paso.disabled, self.btn_auto.disabled = True, True
        tiempo_total = time.time() - self.tiempo_inicio_torneo

        prom_A = sum(self.stats['turnosA'])/len(self.stats['turnosA']) if self.stats['turnosA'] else 0
        prom_B = sum(self.stats['turnosB'])/len(self.stats['turnosB']) if self.stats['turnosB'] else 0
        pct_A = (self.stats['vA'] / self.max_rondas) * 100
        pct_B = (self.stats['vB'] / self.max_rondas) * 100
        ms_A = (self.stats['tiempoA'] / self.stats['movsA']) * 1000 if self.stats['movsA'] else 0
        ms_B = (self.stats['tiempoB'] / self.stats['movsB']) * 1000 if self.stats['movsB'] else 0

        with self.out_log:
            print("\n" + "="*60)
            print("🏁 REPORTE FINAL DEL TORNEO")
            print("="*60)
            print(f"⏱️ Tiempo Total de Ejecución: {tiempo_total:.2f} segundos")
            print(f"\n📊 VICTORIAS:")
            print(f"   • {self.nombreA}: {self.stats['vA']} ({pct_A:.1f}%)")
            print(f"   • {self.nombreB}: {self.stats['vB']} ({pct_B:.1f}%)")
            print(f"   • Empates: {self.stats['empates']}")
            print(f"\n🎯 TURNOS PARA GANAR (PROMEDIO):")
            print(f"   • {self.nombreA}: {prom_A:.2f} turnos")
            print(f"   • {self.nombreB}: {prom_B:.2f} turnos")
            print(f"\n⚡ TIEMPO DE CÓMPUTO:")
            print(f"   • {self.nombreA}: {ms_A:.3f} ms / turno")
            print(f"   • {self.nombreB}: {ms_B:.3f} ms / turno")

            print("\n" + "🌟"*25)
            if self.stats['vA'] > self.stats['vB']: print(f"🏅 GANADOR: ¡{self.nombreA.upper()}!")
            elif self.stats['vB'] > self.stats['vA']: print(f"🏅 GANADOR: ¡{self.nombreB.upper()}!")
            else: print(f"⚖️ ¡EMPATE!")
            print("🌟"*25 + "\n")

        with self.out_vis:
            clear_output(wait=True)
            fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))
            ax1.bar([self.nombreA, self.nombreB, "Empates"], [self.stats['vA'], self.stats['vB'], self.stats['empates']], color=['#2196F3', '#FF9800', '#9E9E9E'])
            ax1.set_title("Rondas Ganadas")
            ax2.bar([self.nombreA, self.nombreB], [prom_A, prom_B], color=['#4CAF50', '#8BC34A'])
            ax2.set_title("Turnos promedio")
            ax3.bar([self.nombreA, self.nombreB], [ms_A, ms_B], color=['#9C27B0', '#E91E63'])
            ax3.set_title("Tiempo (ms/turno)")
            plt.tight_layout()
            plt.show()

# Instanciar el GUI
UI = TorneoPicasFijasUI()

📂 Buscando agentes en '/home/sebastian/Documents/Systems_Engineering/SemesterIX/Sistemas-Inteligentes/Picas&Fijas/Ambientes'
📂 Buscando agentes en '/home/sebastian/Documents/Systems_Engineering/SemesterIX/Sistemas-Inteligentes/Picas&Fijas'
⚠️ La clase 'AgenteJJ&S' en 'Picas_Y_Fijas_Agent.py' fue ignorada porque NO tiene el método 'compute'.
ℹ️ No se encontraron clases válidas con método 'compute' en 'Picas_Y_Fijas_Agent.py'.
✅ Agente cargado con éxito: AgenteJJ&S desde Picas_Y_Fijas_Agent_Compute.py
⚠️ La clase 'AgenteCompetidor' en 'dinoAgent.py' fue ignorada porque NO tiene el método 'compute'.
ℹ️ No se encontraron clases válidas con método 'compute' en 'dinoAgent.py'.
ℹ️ No se encontraron clases válidas con método 'compute' en 'generar_arbol.py'.
⚠️ La clase 'PicasFijasAgent' en 'rival_real.py' fue ignorada porque NO tiene el método 'compute'.
ℹ️ No se encontraron clases válidas con método 'compute' en 'rival_real.py'.
⚠️ La clase '_BaseMemo' en 'rivales_nuevo_ambiente.py' fue ignor